Sequential Chats and Customer Onboarding

## Setup

In [1]:
#from utils import get_openai_api_key
#OPENAI_API_KEY = get_openai_api_key()
#print(OPENAI_API_KEY)
#llm_config = {"model": "gpt-3.5-turbo","api_key":OPENAI_API_KEY}

In [2]:
llm_config = {"model": "llama3","api_type":"ollama"}

## Creating the needed agents

In [3]:

from autogen import ConversableAgent

onboarding_personal_information_agent = ConversableAgent(
    name="onboarding_personal_information_agent",  # Removed spaces
    system_message='''You are a helpful customer onboarding agent.
    You are here to help new customers get started with our product.
    Your job is to gather customer's name and location.
    Do not ask for other information. Return 'TERMINATE' 
    when you have gathered all the information.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
)

onboarding_topic_preference_agent = ConversableAgent(
    name="onboarding_topic_preference_agent",  # Removed spaces
    system_message='''You are a helpful customer onboarding agent.
    You are here to help new customers get started with our product.
    Your job is to gather customer's preferences on news topics.
    Do not ask for other information.
    Return 'TERMINATE' when you have gathered all the information.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
)

customer_engagement_agent = ConversableAgent(
    name="customer_engagement_agent",  # Removed spaces
    system_message='''You are a helpful customer service agent.
    You are here to provide fun for the customer based on the user's
    personal information and topic preferences.
    This could include fun facts, jokes, or interesting stories.
    Make sure to make it engaging and fun!
    Return 'TERMINATE' when you are done.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
    is_termination_msg=lambda msg: "terminate" in msg.get("content").lower(),
)

customer_proxy_agent = ConversableAgent(
    name="customer_proxy_agent",
    llm_config=False,
    code_execution_config=False,
    human_input_mode="ALWAYS",
    is_termination_msg=lambda msg: "terminate" in msg.get("content").lower(),
)


C:\Users\User\anaconda3\Lib\site-packages\autogen\oai\gemini.py:65: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Patching name='__init__', member=<function BedrockClient.__init__ at 0x000001F21B052520>, patched=<function function.__call__ at 0x000001F21B0525C0>
Patching name='_retries', member=5, patched=5
Patching name='cost', member=<function BedrockClient.cost at 0x000001F21B0528E0>, patched=<function function.__call__ at 0x000001F21B052B60>
Patching name='create', member=<function BedrockClient.create at 0x000001F21B052840>, patched=<function function.__call__ at 0x000001F21B052C00>
Patching name='get_usage', member=<function BedrockClient.get_usage at 0x000001F21B052980>, patched=<function function.__call__ at 0x000001F21B052CA0>
Patching name='message_retrieval', member=<function BedrockClient.message_retrieval at 0x000001F21B052660>, patched=<function function.__call__ at 0x000001F21B052D40>
Patching name='parse_custom_params', member=<function BedrockClient.parse_custom_params at 0x000001F21B052700>, patched=<function function.__call__ at 0x000001F21B052DE0>
Patching name='parse_params', 

## Creating tasks

Now, you can craft a series of tasks to facilitate the onboarding process.

In [4]:
chats = [
    {
        "sender": onboarding_personal_information_agent,
        "recipient": customer_proxy_agent,
        "message": "Hello, I'm here to help you get started with our product. Could you tell me your name and location?",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Return the customer information as JSON object only: {'name': '', 'location': ''}",
        },
        "max_turns": 1,
        "clear_history": True
    },
    {
        "sender": onboarding_topic_preference_agent,
        "recipient": customer_proxy_agent,
        "message": "Great! Could you tell me what topics you are interested in reading about?",
        "summary_method": "reflection_with_llm",
        "max_turns": 1,
        "clear_history": False
    },
    {
        "sender": customer_proxy_agent,
        "recipient": customer_engagement_agent,
        "message": "Let's find something fun to read.",
        "max_turns": 1,
        "summary_method": "reflection_with_llm",
    },
]

## Start the onboarding process

**Note**: You might get a slightly different response than what's shown in the video. Feel free to try different inputs, such as name, location, and preferences.

In [5]:
from autogen import initiate_chats

chat_results = initiate_chats(chats)



********************************************************************************
Starting a new chat....

********************************************************************************
onboarding_personal_information_agent (to customer_proxy_agent):

Hello, I'm here to help you get started with our product. Could you tell me your name and location?

--------------------------------------------------------------------------------


C:\Users\User\anaconda3\Lib\site-packages\autogen\agentchat\chat.py:51: UserWarning: Repetitive recipients detected: The chat history will be cleared by default if a recipient appears more than once. To retain the chat history, please set 'clear_history=False' in the configuration of the repeating agent.
  warnings.warn(


Replying as customer_proxy_agent. Provide feedback to onboarding_personal_information_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  Ashika from Vellore


customer_proxy_agent (to onboarding_personal_information_agent):

Ashika from Vellore

--------------------------------------------------------------------------------

********************************************************************************
Starting a new chat....

********************************************************************************
onboarding_topic_preference_agent (to customer_proxy_agent):

Great! Could you tell me what topics you are interested in reading about?
Context: 
{'name': 'Ashika', 'location': 'Vellore'}

--------------------------------------------------------------------------------


Replying as customer_proxy_agent. Provide feedback to onboarding_topic_preference_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  Testing


customer_proxy_agent (to onboarding_topic_preference_agent):

Testing

--------------------------------------------------------------------------------

********************************************************************************
Starting a new chat....

********************************************************************************
customer_proxy_agent (to customer_engagement_agent):

Let's find something fun to read.
Context: 
{'name': 'Ashika', 'location': 'Vellore'}
No specific topics were mentioned, but the context provided details about the person, Ashika, who is located in Vellore.

--------------------------------------------------------------------------------
customer_engagement_agent (to customer_proxy_agent):

Hello Ashika! It's so great to chat with you!

Since you're from Vellore, I thought I'd share a fun fact about Tamil Nadu, your state! Did you know that Vellore is home to the largest Christian settlement in Asia? It's a fascinating piece of history!

To make it 

## Print out the summary

In [17]:
summary=[]
for chat_result in chat_results:
    print(chat_result.summary)
    summary.append(chat_result.summary)
    print("\n")

{'content': "{'name': 'Ashika', 'location': 'Vellore'}", 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}


{'content': 'No specific topics were mentioned, but the context provided details about the person, Ashika, who is located in Vellore.', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}


{'content': "A fun fact about Vellore's Christian settlement, the Vellore Mutiny story, and a joke about a Tamil Nadu cow.", 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}




In [7]:
summary[0]["content"]

"{'name': 'Ashika', 'location': 'Vellore'}"

In [8]:
#!pip install pandas

In [9]:
import pandas as pd
import ast

# Input string
#data_str = "{'name': 'Ramisha', 'location': 'Coimbatore'}"
data_str=summary[0]["content"]
# Convert string to dictionary
data_dict = ast.literal_eval(data_str)

# Convert dictionary to DataFrame
df = pd.DataFrame([data_dict])

df


,name,location
0,Ashika,Vellore


In [10]:
df["TopicPreference"]=summary[1]["content"]

In [11]:
df

,name,location,TopicPreference
0,Ashika,Vellore,"No specific topics were mentioned, but the con..."


In [12]:
df["Enagement"]=summary[2]["content"]

In [13]:
df

,name,location,TopicPreference,Enagement
0,Ashika,Vellore,"No specific topics were mentioned, but the con...",A fun fact about Vellore's Christian settlemen...


In [14]:
#!pip install gspread
#!pip install oauth2client

In [15]:
def add_leads_to_google_sheet(df):
    import gspread
    from oauth2client.service_account import ServiceAccountCredentials

    # Define the scope for Google Sheets API
    scope = ['https://www.googleapis.com/auth/spreadsheets']

    # Add credentials to the account
    creds = ServiceAccountCredentials.from_json_keyfile_name(
        'websiteleadchatgptapi-a46f50ae1507.json', scope
    )

    # Authorize the client
    client = gspread.authorize(creds)

    # Specify the Google Sheet ID
    sheet_id = '1w-19wp0W2nRV5Pde5gim6kKlCoPTjv6mT3A8lAdNrcM'

    # Open the spreadsheet using its ID
    spreadsheet = client.open_by_key(sheet_id)

    # Select the first worksheet (index 0)
    worksheet = spreadsheet.get_worksheet(0)

    # Append each row of the DataFrame to the worksheet
    for _, row in df.iterrows():
        # Convert row to a list and append to the sheet
        worksheet.append_row(row.tolist())

    print("Data added successfully!")
    return "added"


In [18]:
#add_leads_to_google_sheet(df)

## Print out the cost

In [19]:
for chat_result in chat_results:
    print(chat_result.cost)
    print("\n")

{'usage_including_cached_inference': {'total_cost': 0, 'llama3': {'cost': 0, 'prompt_tokens': 67, 'completion_tokens': 16, 'total_tokens': 83}}, 'usage_excluding_cached_inference': {'total_cost': 0, 'llama3': {'cost': 0, 'prompt_tokens': 67, 'completion_tokens': 16, 'total_tokens': 83}}}


{'usage_including_cached_inference': {'total_cost': 0, 'llama3': {'cost': 0, 'prompt_tokens': 70, 'completion_tokens': 27, 'total_tokens': 97}}, 'usage_excluding_cached_inference': {'total_cost': 0, 'llama3': {'cost': 0, 'prompt_tokens': 70, 'completion_tokens': 27, 'total_tokens': 97}}}


{'usage_including_cached_inference': {'total_cost': 0, 'llama3': {'cost': 0, 'prompt_tokens': 401, 'completion_tokens': 210, 'total_tokens': 611}}, 'usage_excluding_cached_inference': {'total_cost': 0, 'llama3': {'cost': 0, 'prompt_tokens': 401, 'completion_tokens': 210, 'total_tokens': 611}}}


